<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# Green Thumbs-Up Superset (PACs)

This notebook is a test sample to upload all the CSV to firebase.

Sai, Lindsey and I still need to agree where the data will be uploaded (Firebase project).


# Imports

In [1]:
import pandas as pd
import re
import os
from rapidfuzz import process, fuzz
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Merge Data From FEC

In [ ]:
# Folder where your txt files are
folder = r"C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 8"

files = ["2020", "2022", "2024", "2026"]

all_fec = []

for year in files:
    path = os.path.join(folder, f"{year}.txt")
    
    df = pd.read_csv(
        path,
        sep="|",
        header=None,
        dtype=str,
        encoding="latin1"
    )
    
    df.columns = [
        "CMTE_ID",
        "CMTE_NM",
        "TRES_NM",
        "CMTE_ST1",
        "CMTE_ST2",
        "CMTE_CITY",
        "CMTE_ST",
        "CMTE_ZIP",
        "CMTE_DSGN",
        "CMTE_TP",
        "CMTE_PTY_AFFILIATION",
        "CMTE_FILING_FREQ",
        "ORG_TP",
        "CONNECTED_ORG_NM",
        "CAND_ID"
    ]
    
    df["ELECTION_CYCLE"] = year  # track source cycle
    
    all_fec.append(df)

# Combine all cycles
fec_master = pd.concat(all_fec, ignore_index=True)

# Remove duplicate CMTE_IDs across cycles
fec_master = fec_master.drop_duplicates(subset=["CMTE_ID"])

# Save unified master
fec_master.to_csv("fec_committee_master_2020_2026.csv", index=False)

fec_master.head()

# File Paths

In [2]:
TASK8_DIR = r"C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 8"

SNOWFLAKE_CSV = os.path.join(TASK8_DIR, "snowflake_company_pac_contributions.csv")
FEC_MASTER_CSV = os.path.join(TASK8_DIR, "fec_committee_master_2020_2026.csv")

OUT_MATCHED = os.path.join(TASK8_DIR, "task8_private_pac_matches.csv")
OUT_SUMMARY = os.path.join(TASK8_DIR, "task8_private_pac_match_summary.csv")
OUT_REVIEW  = os.path.join(TASK8_DIR, "task8_private_pac_fuzzy_review.csv")
OUT_TOP_UNMATCHED = os.path.join(TASK8_DIR, "task8_private_pac_top_unmatched_committees.csv")

# Load CSVs

In [3]:
sf = pd.read_csv(SNOWFLAKE_CSV, dtype=str)
fec = pd.read_csv(FEC_MASTER_CSV, dtype=str)

print("Snowflake rows:", len(sf))
print("FEC master rows:", len(fec))
print("Snowflake columns:", list(sf.columns))
print("FEC columns:", list(fec.columns))

sf.head()

Snowflake rows: 242230
FEC master rows: 35169
Snowflake columns: ['TICKER', 'ELECTION_CYCLE', 'COMMITTEE_NAME', 'COMMITTEE_TYPE_FULL', 'PARTY_NAME_FULL', 'RECEIPT_AMOUNT']
FEC columns: ['CMTE_ID', 'CMTE_NM', 'TRES_NM', 'CMTE_ST1', 'CMTE_ST2', 'CMTE_CITY', 'CMTE_ST', 'CMTE_ZIP', 'CMTE_DSGN', 'CMTE_TP', 'CMTE_PTY_AFFILIATION', 'CMTE_FILING_FREQ', 'ORG_TP', 'CONNECTED_ORG_NM', 'CAND_ID', 'ELECTION_CYCLE']


,TICKER,ELECTION_CYCLE,COMMITTEE_NAME,COMMITTEE_TYPE_FULL,PARTY_NAME_FULL,RECEIPT_AMOUNT
0,TSCO,2022,JOE MORELLE FOR CONGRESS,House,Democrat,1000.000000
1,T,2022,DR. ASIF MAHMOOD VICTORY FUND,PAC - Nonqualified,Democrat,6000.000000
2,T,2022,VICENTE GONZALEZ VICTORY FUND,PAC - Nonqualified,Democrat,3000.000000
3,EIX,2022,DEBBIE DINGELL FOR CONGRESS,House,Democrat,2500.000000
4,DAL,2022,JASON SMITH FOR CONGRESS,House,Republican,2500.000000


# Scope Filter

In [4]:
ELECTION_COL = "ELECTION_CYCLE"

sf[ELECTION_COL] = pd.to_numeric(sf[ELECTION_COL], errors="coerce")
sf = sf[sf[ELECTION_COL].isin([2020, 2022, 2024, 2026])].copy()

print("Snowflake rows after cycle filter:", len(sf))
sf[[ELECTION_COL]].value_counts(dropna=False).sort_index()


Snowflake rows after cycle filter: 153082


ELECTION_CYCLE
2020              49633
2022              41174
2024              45958
2026              16317
Name: count, dtype: int64

# Normalize Committee Names

In [5]:
STOP = {
    "THE","AND","OF","FOR","TO","A","AN",
    "INC","INCORPORATED","CORP","CORPORATION","CO","COMPANY",
    "LLC","LTD","LIMITED","LP","LLP","HOLDING","HOLDINGS","GROUP"
}

def norm_name(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x).upper()
    x = re.sub(r"[^A-Z0-9 ]+", " ", x)  # remove punctuation
    x = re.sub(r"\s+", " ", x).strip() # collapse whitespace
    toks = [t for t in x.split() if t not in STOP]
    return " ".join(toks)

# Adjust if your column name differs (should be COMMITTEE_NAME)
CMTE_COL = "COMMITTEE_NAME"

sf["cmte_norm"] = sf[CMTE_COL].map(norm_name)
fec["cmte_norm"] = fec["CMTE_NM"].map(norm_name)

# Drop blanks
sf = sf[sf["cmte_norm"] != ""].copy()
fec = fec[fec["cmte_norm"] != ""].copy()

print("Snowflake rows (non-empty committee):", len(sf))
print("FEC rows (non-empty committee):", len(fec))

Snowflake rows (non-empty committee): 153082
FEC rows (non-empty committee): 35165


# Exact Match

In [6]:
# Keep 1 row per normalized committee name on FEC side
fec_key = fec.drop_duplicates("cmte_norm").set_index("cmte_norm")

out = sf.join(
    fec_key[["CMTE_ID","CMTE_NM","CONNECTED_ORG_NM","CMTE_TP","ORG_TP"]],
    on="cmte_norm",
    how="left",
    rsuffix="_fec"
)

out["match_method"] = out["CMTE_ID"].notna().map(lambda b: "NAME_EXACT" if b else "NO_MATCH")
out["match_score"]  = out["CMTE_ID"].notna().map(lambda b: 100 if b else None)

print("Exact matched rows:", out["CMTE_ID"].notna().sum())
print("Still unmatched:", out["CMTE_ID"].isna().sum())

out.head()

Exact matched rows: 137440
Still unmatched: 15642


,TICKER,ELECTION_CYCLE,COMMITTEE_NAME,COMMITTEE_TYPE_FULL,PARTY_NAME_FULL,RECEIPT_AMOUNT,cmte_norm,CMTE_ID,CMTE_NM,CONNECTED_ORG_NM,CMTE_TP,ORG_TP,match_method,match_score
0,TSCO,2022,JOE MORELLE FOR CONGRESS,House,Democrat,1000.000000,JOE MORELLE CONGRESS,C00675108,JOE MORELLE FOR CONGRESS,MORELLE VICTORY FUND,H,NaN,NAME_EXACT,100.0
1,T,2022,DR. ASIF MAHMOOD VICTORY FUND,PAC - Nonqualified,Democrat,6000.000000,DR ASIF MAHMOOD VICTORY FUND,C00813451,DR. ASIF MAHMOOD VICTORY FUND,NaN,N,NaN,NAME_EXACT,100.0
2,T,2022,VICENTE GONZALEZ VICTORY FUND,PAC - Nonqualified,Democrat,3000.000000,VICENTE GONZALEZ VICTORY FUND,C00825257,VICENTE GONZALEZ VICTORY FUND,NONE,N,NaN,NAME_EXACT,100.0
3,EIX,2022,DEBBIE DINGELL FOR CONGRESS,House,Democrat,2500.000000,DEBBIE DINGELL CONGRESS,C00558213,DEBBIE DINGELL FOR CONGRESS,WOLVERINE VICTORY FUND,H,NaN,NAME_EXACT,100.0
4,DAL,2022,JASON SMITH FOR CONGRESS,House,Republican,2500.000000,JASON SMITH CONGRESS,C00541862,JASON SMITH FOR CONGRESS,SMITH VICTORY,H,NaN,NAME_EXACT,100.0


# Fuzzy Match Residuals

In [7]:
unmatched = out[out["CMTE_ID"].isna()].copy()

# Build candidate list from FEC
choices = fec_key.reset_index()[["cmte_norm","CMTE_ID","CMTE_NM","CONNECTED_ORG_NM","CMTE_TP","ORG_TP"]].copy()
choice_list = choices["cmte_norm"].tolist()
choice_map = choices.set_index("cmte_norm").to_dict(orient="index")

def best_fuzzy(q: str):
    hit = process.extractOne(q, choice_list, scorer=fuzz.token_set_ratio)
    if not hit:
        return None, None
    return hit[0], hit[1]

unmatched[["best_norm","fuzzy_score"]] = unmatched["cmte_norm"].apply(
    lambda s: pd.Series(best_fuzzy(s))
)

AUTO = 95
REVIEW = 90

def bucket(score):
    if pd.isna(score): 
        return "NO_MATCH"
    if score >= AUTO: 
        return "FUZZY_HIGH"
    if score >= REVIEW: 
        return "FUZZY_REVIEW"
    return "NO_MATCH"

unmatched["match_method"] = unmatched["fuzzy_score"].apply(bucket)

def attach(row):
    if row["match_method"] in {"FUZZY_HIGH","FUZZY_REVIEW"}:
        info = choice_map.get(row["best_norm"], {})
        row["CMTE_ID"] = info.get("CMTE_ID")
        row["CMTE_NM"] = info.get("CMTE_NM")
        row["CONNECTED_ORG_NM"] = info.get("CONNECTED_ORG_NM")
        row["CMTE_TP"] = info.get("CMTE_TP")
        row["ORG_TP"] = info.get("ORG_TP")
        row["match_score"] = row["fuzzy_score"]
    return row

unmatched = unmatched.apply(attach, axis=1)

print("Fuzzy HIGH:", (unmatched["match_method"] == "FUZZY_HIGH").sum())
print("Fuzzy REVIEW:", (unmatched["match_method"] == "FUZZY_REVIEW").sum())
print("Still NO_MATCH:", (unmatched["match_method"] == "NO_MATCH").sum())

unmatched.head()

Fuzzy HIGH: 8774
Fuzzy REVIEW: 611
Still NO_MATCH: 6257


,TICKER,ELECTION_CYCLE,COMMITTEE_NAME,COMMITTEE_TYPE_FULL,PARTY_NAME_FULL,RECEIPT_AMOUNT,cmte_norm,CMTE_ID,CMTE_NM,CONNECTED_ORG_NM,CMTE_TP,ORG_TP,match_method,match_score,best_norm,fuzzy_score
9,SO,2022,START RISING PAC,PAC - Qualified,Republican,2500.000000,START RISING PAC,C00764761,START RISING POLITICAL ACTION COMMITTEE OR STA...,LETLOW,N,NaN,FUZZY_HIGH,100.0,START RISING POLITICAL ACTION COMMITTEE OR STA...,100.0
77,SYY,2022,TIM SCOTT FOR AMERICA,Senate,Republican,2500.000000,TIM SCOTT AMERICA,C00837872,AMERICA,NaN,P,NaN,FUZZY_HIGH,100.0,AMERICA,100.0
80,IVZ,2022,KS COMMITTEE,PAC - Nonqualified,Democrat,1000.000000,KS COMMITTEE,C00848572,THE COMMITTEE,NONE,P,NaN,FUZZY_HIGH,100.0,COMMITTEE,100.0
96,SYK,2022,PETERS LEADERSHIP FUND,PAC - Nonqualified,Democrat,5000.000000,PETERS LEADERSHIP FUND,C00811968,THE LEADERSHIP FUND,NONE,X,NaN,FUZZY_HIGH,100.0,LEADERSHIP FUND,100.0
114,LUV,2020,CENTER AISLE PAC,PAC - Qualified,Democrat,1000.000000,CENTER AISLE PAC,C00488585,CENTER; THE,NaN,N,NaN,FUZZY_HIGH,100.0,CENTER,100.0


# Consolidate Final + Summary

In [9]:
matched_exact = out[out["CMTE_ID"].notna()].copy()
final = pd.concat([matched_exact, unmatched], ignore_index=True)

summary = (
    final.groupby("match_method")
         .size()
         .reset_index(name="rows")
         .sort_values("rows", ascending=False)
)

total_rows = len(final)
matched_rows = final["CMTE_ID"].notna().sum()
match_rate = matched_rows / total_rows if total_rows else 0

print(summary)
print("Total rows:", total_rows)
print("Matched rows:", matched_rows)
print("Match rate:", round(match_rate, 4))

   match_method    rows
2    NAME_EXACT  137440
0    FUZZY_HIGH    8774
3      NO_MATCH    6257
1  FUZZY_REVIEW     611
Total rows: 153082
Matched rows: 146825
Match rate: 0.9591


# Top Unmatched Committees (Frequency)

In [10]:
top_unmatched = (
    final[final["CMTE_ID"].isna()]
    .groupby(CMTE_COL)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .head(50)
)

top_unmatched

,COMMITTEE_NAME,rows
27,CMR WA PAC,426
4,"ANDY BARR FOR SENATE, INC.",379
48,JOHN CURTIS FOR UTAH,333
1,ADF PAC,288
18,BUDDY CARTER FOR SENATE,276
31,DIGNITY OF WORK PAC,264
8,ASHLEY FOR IOWA,237
96,WAGING PEACE,215
61,OHIO BELIEF PAC,211
10,BANKS FOR SENATE,196


# Save Outputs

In [11]:
final.to_csv(OUT_MATCHED, index=False)
summary.to_csv(OUT_SUMMARY, index=False)

review = final[final["match_method"] == "FUZZY_REVIEW"].copy()
review.to_csv(OUT_REVIEW, index=False)

top_unmatched.to_csv(OUT_TOP_UNMATCHED, index=False)

print("Saved:", OUT_MATCHED)
print("Saved:", OUT_SUMMARY)
print("Saved:", OUT_REVIEW)
print("Saved:", OUT_TOP_UNMATCHED)

Saved: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 8\task8_private_pac_matches.csv
Saved: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 8\task8_private_pac_match_summary.csv
Saved: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 8\task8_private_pac_fuzzy_review.csv
Saved: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 8\task8_private_pac_top_unmatched_committees.csv


In [12]:
final.groupby("match_method").size()

match_method
FUZZY_HIGH        8774
FUZZY_REVIEW       611
NAME_EXACT      137440
NO_MATCH          6257
dtype: int64

In [13]:
final["COMMITTEE_TYPE_FULL"].value_counts()

COMMITTEE_TYPE_FULL
House                                                        71636
PAC - Qualified                                              29110
Senate                                                       24389
PAC - Nonqualified                                           14768
Party - Qualified                                             5147
Hybrid PAC (with Non-Contribution Account) - Nonqualified     2422
Presidential                                                  2146
Hybrid PAC (with Non-Contribution Account) - Qualified        1794
Super PAC (Independent Expenditure-Only)                      1369
Party - Nonqualified                                           297
Single Candidate Independent Expenditure                         4
Name: count, dtype: int64